In [1]:
import os, sys, math
import numpy as np
import inspect
from sys import stdout
import glob
import tempfile
from openmm import *
from openmm.app import *
from openmm.unit import *
from openmmtools import testsystems, states, mcmc
from openmmtools import forces, alchemy
from openmmtools.multistate import ReplicaExchangeSampler, MultiStateReporter
import mdtraj
import logging
import datetime
import warnings

In [2]:
import barnaba as bb
from barnaba import definitions
from barnaba.nucleic import Nucleic

### paramters

In [3]:
prmtopfile = "/Users/takabak/work/espaloma_rna/validation/scripts/sample/aaaa/rna.ol3/00_setup/input.prmtop"
inpcrdfile = "/Users/takabak/work/espaloma_rna/validation/scripts/sample/aaaa/rna.ol3/00_setup/input.inpcrd"

In [4]:
system_options = {}
system_options['nonbondedMethod'] = PME
#system_options['ewaldErrorTolerance'] = 0.0005
system_options['nonbondedCutoff'] = 9 * angstroms
system_options['switchDistance'] = 8 * angstroms
system_options['rigidWater'] = True
system_options['removeCMMotion'] = False
system_options['constraints'] = HBonds
system_options['hydrogenMass'] = 3.5 * amu

In [5]:
default_pressure = 1 * atmosphere
default_temperature = 300 * kelvin
default_timestep = 4 * femtosecond
#default_collision_rate = 1
default_swap_scheme = 'swap-all'
default_stes_per_replica = 250
default_number_of_iterations = 100
default_checkpoint_interval = 5

### create system

In [6]:
prmtop = AmberPrmtopFile(prmtopfile)
inpcrd = AmberInpcrdFile(inpcrdfile)

In [7]:
system = prmtop.createSystem(**system_options)

In [8]:
system.getDefaultPeriodicBoxVectors()

[Quantity(value=Vec3(x=4.879659500000001, y=0.0, z=0.0), unit=nanometer),
 Quantity(value=Vec3(x=0.0, y=4.377179600000001, z=0.0), unit=nanometer),
 Quantity(value=Vec3(x=0.0, y=0.0, z=3.9527219000000002), unit=nanometer)]

In [9]:
# define integrator
integrator = LangevinMiddleIntegrator(default_temperature, 1/picosecond, default_timestep)

In [10]:
# add barostat
system.addForce(MonteCarloBarostat(default_pressure, default_temperature))

4

In [11]:
forces = list(system.getForces())
forces

[<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x13835dd50> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x13835db40> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x13835de70> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x13835ded0> >,
 <openmm.openmm.MonteCarloBarostat; proxy of <Swig Object of type 'OpenMM::MonteCarloBarostat *' at 0x13835df60> >]

### torsion atom index

In [12]:
"""
Calculate 6 torsion angles (α, β, γ, δ, ε, and ζ) around the consecutive chemical bonds, chi (χ) quantifying the relative base/sugar orientation, plus the sugar pucker.
https://x3dna.org/highlights/pseudo-torsions-to-simplify-the-representation-of-dna-rna-backbone-conformation
"""   

p = mdtraj.load_prmtop(prmtopfile)
top = mdtraj.load(inpcrdfile, top=p)

n = Nucleic(topology=top.topology)
idx, r =  n.get_bb_torsion_idx()

# remove undefined backbones which are indexed as 0 and the first 4 index that defines the γ torsion angle
#idx = idx.flatten()
#idx = idx[idx != 0][4:]
#atom_indices = np.unique(idx).tolist()

# Skipping unknown residue Na+4 
# Skipping unknown residue Na+5 
# Skipping unknown residue Na+6 
# Skipping unknown residue Na+7 
# Skipping unknown residue Na+8 
# Skipping unknown residue Na+9 
# Skipping unknown residue Na+10 
# Skipping unknown residue Cl-11 
# Skipping unknown residue Cl-12 
# Skipping unknown residue Cl-13 
# Skipping unknown residue Cl-14 


In [13]:
idx = idx.reshape(28, 4)[3:]
idx = np.array(idx)

In [14]:
idx

array([[  2,   5,  24,  30],
       [  5,  24,  30,  31],
       [ 24,  30,  31,  34],
       [  7,   8,  10,  23],
       [ 30,  31,  34,  35],
       [ 31,  34,  35,  38],
       [ 34,  35,  38,  57],
       [ 35,  38,  57,  63],
       [ 38,  57,  63,  64],
       [ 57,  63,  64,  67],
       [ 40,  41,  43,  56],
       [ 63,  64,  67,  68],
       [ 64,  67,  68,  71],
       [ 67,  68,  71,  90],
       [ 68,  71,  90,  96],
       [ 71,  90,  96,  97],
       [ 90,  96,  97, 100],
       [ 73,  74,  76,  89],
       [ 96,  97, 100, 101],
       [ 97, 100, 101, 104],
       [100, 101, 104, 123],
       [101, 104, 123, 129],
       [  0,   0,   0,   0],
       [  0,   0,   0,   0],
       [106, 107, 109, 122]])

In [20]:
torsion_indices = []

for force in forces:
    name = force.__class__.__name__
    if "Torsion" in name:
        for i in range(force.getNumTorsions()):
            id1, id2, id3, id4, periodicity, phase, k = force.getTorsionParameters(i)
            #print(i, force.getTorsionParameters(i))
            x = np.array([id1, id2, id3, id4])
            for _idx in idx:
                c = _idx == x
                if c.all():
                    torsion_indices.append(i)
                    #print(i, force.getTorsionParameters(i))

In [23]:
#torsion_indices

### create alchemy system

In [24]:
alchemy.AlchemicalRegion?

Init signature:
alchemy.AlchemicalRegion(
    alchemical_atoms=None,
    alchemical_bonds=None,
    alchemical_angles=None,
    alchemical_torsions=None,
    annihilate_electrostatics=True,
    annihilate_sterics=False,
    softcore_alpha=0.5,
    softcore_a=1,
    softcore_b=1,
    softcore_c=6,
    softcore_beta=0.0,
    softcore_d=1,
    softcore_e=1,
    softcore_f=2,
    name=None,
)
Docstring:     
Alchemical region.

This is a namedtuple used to tell the AbsoluteAlchemicalFactory which
region of the system to alchemically modify and how.

Parameters
----------
alchemical_atoms : list of int, optional
    List of atoms to be designated for which the nonbonded forces (both
    sterics and electrostatics components) have to be alchemically
    modified (default is None).
alchemical_bonds : bool or list of int, optional
    If a list of bond indices are specified, these HarmonicBondForce
    entries are softened with 'lambda_bonds'. If set to True, this list
    is auto-generated to

In [25]:
alchemy.AlchemicalRegion()

AlchemicalRegion(alchemical_atoms=None, alchemical_bonds=None, alchemical_angles=None, alchemical_torsions=None, annihilate_electrostatics=True, annihilate_sterics=False, softcore_alpha=0.5, softcore_a=1, softcore_b=1, softcore_c=6, softcore_beta=0.0, softcore_d=1, softcore_e=1, softcore_f=2, name=None)

In [26]:
# define alchemical region and thermodynamic state
#alchemical_region = alchemy.AlchemicalRegion(alchemical_atoms=atom_indices, alchemical_torsions=True)
alchemical_region = alchemy.AlchemicalRegion(alchemical_torsions=torsion_indices)
factory = alchemy.AbsoluteAlchemicalFactory()
alchemical_system = factory.create_alchemical_system(system, alchemical_region)

In [34]:
temp = default_temperature
#protocol = {'temperature':           [temp, temp, temp, temp, temp, temp], \
#            'lambda_electrostatics': [1.00, 1.00, 1.00, 1.00, 1.00, 1.00], \
#            'lambda_sterics':        [1.00, 1.00, 1.00, 1.00, 1.00, 1.00], \
#            'lambda_torsions':       [1.00, 0.75, 0.50, 0.25, 0.00, 0.00]}
protocol = {'temperature':           [temp, temp, temp, temp, temp, temp], \
            'lambda_torsions':       [1.00, 0.75, 0.50, 0.25, 0.00, 0.00]}

### define alchemical state and sampler state

In [35]:
alchemical_state = alchemy.AlchemicalState.from_system(alchemical_system)

In [36]:
thermodynamics_states = states.create_thermodynamic_state_protocol(alchemical_system, protocol=protocol, composable_states=[alchemical_state])

In [37]:
sampler_states = states.SamplerState(positions=inpcrd.positions, box_vectors=inpcrd.boxVectors)

In [38]:
simulation = ReplicaExchangeSampler(number_of_iterations=default_number_of_iterations, replica_mixing_scheme=default_swap_scheme, online_analysis_interval=None)
reporter = MultiStateReporter(storage='enhanced.nc', checkpoint_interval=default_checkpoint_interval)

/bin/sh: nvidia-smi: command not found


In [39]:
# remove old storge if exists
storage_file = 'enhanced.nc'
n = glob.glob(storage_file + '.nc')
if os.path.exists(storage_file):
    print('{} already exists. File will be renamed but checkpoint files will be deleted'.format(storage_file))
    os.remove('enhanced_checkpoint.nc')
    os.rename(storage_file, storage_file + "{}".format(str(len(n))))

enhanced.nc already exists. File will be renamed but checkpoint files will be deleted


In [40]:
simulation.create(thermodynamics_states, sampler_states=sampler_states, storage=reporter)

Please cite the following:

        Friedrichs MS, Eastman P, Vaidyanathan V, Houston M, LeGrand S, Beberg AL, Ensign DL, Bruns CM, and Pande VS. Accelerating molecular dynamic simulations on graphics processing unit. J. Comput. Chem. 30:864, 2009. DOI: 10.1002/jcc.21209
        Eastman P and Pande VS. OpenMM: A hardware-independent framework for molecular simulations. Comput. Sci. Eng. 12:34, 2010. DOI: 10.1109/MCSE.2010.27
        Eastman P and Pande VS. Efficient nonbonded interactions for molecular dynamics on a graphics processing unit. J. Comput. Chem. 31:1268, 2010. DOI: 10.1002/jcc.21413
        Eastman P and Pande VS. Constant constraint matrix approximation: A robust, parallelizable constraint method for molecular simulations. J. Chem. Theor. Comput. 6:434, 2010. DOI: 10.1021/ct900463w
        Chodera JD and Shirts MR. Replica exchange and expanded ensemble simulations as Gibbs multistate: Simple improvements for enhanced mixing. J. Chem. Phys., 135:194110, 2011. DOI:10.1063/

In [42]:
# Just to check if the performance is better using this - for Openmm <= 7.7
from openmmtools.utils import get_fastest_platform
from openmmtools.cache import ContextCache
platform = get_fastest_platform(minimum_precision='mixed')
simulation.energy_context_cache = ContextCache(capacity=None, time_to_live=None, platform=platform)
simulation.sampler_context_cache = ContextCache(capacity=None, time_to_live=None, platform=platform)

In [ ]:
#simulation.run()